In [ ]:
using MPSDynamics, ChaosTools, ProgressBars, Statistics
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/config/requirements.jl")

In [ ]:
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞
ϕ_quantum=1/2

#### Caracterization of the circuit #####

ω_a = 25.338776456203686
ω_b = 2*ω_a


φ_a = 0.11
Z_a = 2*ϕ_quantum^2*φ_a^2
C_a_inv = Z_a*ω_a
E_Ca = C_a_inv/2

φ_b = 0.204
Z_b = 2*ϕ_quantum^2*φ_b^2
C_b_inv = Z_b*ω_b
E_Cb = C_b_inv/2

E_J = 37.12*2*pi*1.5


##Drive
ω_d=ω_b
ω_p=2ω_a-ω_d

buffer_phase = 0.0
pump_phase = 0.0;

##Env
κ_b=π/10.4
κ_a = 0

# Poincaré sections

In [ ]:
ϵ_p_list = range(0.58, 1.1, 100)#[0.58, 0.6894736842105263, 0.8263157894736842]#

t_max = 50/κ_b
t_min = 0.0
tspan = (0.0, t_max) # Time span for the simulation
dt = 1 / (50 * ω_a) # Time step for the simulation

dt_save=2pi/(30*sqrt(2)*ω_a)
t_min_save = t_max - 1000*dt_save
t_save = max(t_min_save,0):dt_save:t_max


b_0=0.0
environment = MeanField(κ_a, κ_b)
system = SystemATS_ladder(ω_a, ω_b, E_J, E_Ca, E_Cb, 0.0, b_0)

environment_prepared = prepare(system,environment);
t_list = collect(range(max(t_min_save,0),step = dt_save,stop = t_max));

Im_a_list=range(-4,4,10)
Re_a_list=range(-2.5, 2.5 ,25)
Final_all_a = Dict{Float64,Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}}()
Final_all_b = Dict{Float64,Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}}()
Final_all_a_cat = Dict{Float64,Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}}()
Final_all_b_cat = Dict{Float64,Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}}()

Threads.@threads for ϵ_p in ϵ_p_list
    g2 = besselj(1,ϵ_p)*E_J*φ_a^2*φ_b # (Leon's paper)
    ϵ_d=10*g2
    drive = DriveTwoModesBasics(ϵ_d, ω_d, buffer_phase, ϵ_p, ω_p, pump_phase)
    drive_prepared = prepare(drive)
    Final_mode_a = Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}()
    Final_mode_b = Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}()
    Final_mode_a_cat = Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}()
    Final_mode_b_cat = Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}()
    for i in 1:length(Re_a_list)
        re_i = Re_a_list[i]
        for j in 1:length(Im_a_list)
            im_j = Im_a_list[j]
            a_0_j = (re_i + 1im*im_j)

            system = SystemATS_ladder(ω_a, ω_b, E_J, E_Ca, E_Cb, a_0_j, b_0)
            system_prepared = prepare(system)

            u_0 = get_init(system_prepared, environment_prepared)
            p = SimulationParams(system_prepared, drive_prepared, environment_prepared)

            a, b, t_list = default_solver(u_0, (t_min, t_max), p, dt, t_save = t_save, equation = default_equation!)

            a_rot = a.*exp.(1im*ω_a*t_list)*1im*φ_a #90° rotation
            b_rot = b.*exp.(1im*ω_b*t_list)*1im*φ_b #90° rotation

            if abs(maximum(real(a_rot)))<0.2 && abs(minimum(real(a_rot)))>0.15 && maximum(abs.(imag(a_rot)))<0.05
                Final_mode_a_cat[(re_i,im_j)] = a_rot
                Final_mode_b_cat[(re_i,im_j)] = b_rot
            else 
                Final_mode_a[(re_i,im_j)] = a_rot
                Final_mode_b[(re_i,im_j)] = b_rot
            end    
        end
    end
    Final_all_a[ϵ_p] = Final_mode_a
    Final_all_b[ϵ_p] = Final_mode_b
    Final_all_a_cat[ϵ_p] = Final_mode_a_cat
    Final_all_b_cat[ϵ_p] = Final_mode_b_cat
end;

In [ ]:
# Non-linearity of one red (non-cat) trajectory as a function of ϵ_p.
# The only non-linearity in the equations of motion is the junction term cos(θ) - 1, with
#   θ = (a+a*)/√2 · √Z_a/ϕ_quantum + (b+b*)/√2 · √Z_b/ϕ_quantum = 2 Re(φ_a a + φ_b b).
# Final_all_a / Final_all_b store a_rot = i φ_a a e^{iω_a t}, so φ_a a = -i a_rot e^{-iω_a t}.
# Beyond-quadratic content of the junction: nl(t) = cos θ - 1 + θ²/2.

ref_ic = (0.0, 0.0)   # (Re a_0, Im a_0): for each ϵ_p, the red trajectory whose initial condition is closest to this one is used

ϵ_list = collect(ϵ_p_list)
θ_max  = fill(NaN, length(ϵ_list))   # max_t |θ|
nl_rel = fill(NaN, length(ϵ_list))   # <|cos θ - 1 + θ²/2|> / <θ²/2>

for (k, ϵ_p) in enumerate(ϵ_list)
    red_a = Final_all_a[ϵ_p]
    red_b = Final_all_b[ϵ_p]
    isempty(red_a) && continue   # no red trajectory at this ϵ_p

    key = argmin(kk -> (kk[1] - ref_ic[1])^2 + (kk[2] - ref_ic[2])^2, collect(keys(red_a)))
    t   = collect(t_save)
    a_rot = ComplexF64.(red_a[key])
    b_rot = ComplexF64.(red_b[key])

    θ  = @. 2 * real(-1im * (a_rot * exp(-1im * ω_a * t) + b_rot * exp(-1im * ω_b * t)))
    nl = @. cos(θ) - 1 + θ^2 / 2

    θ_max[k]  = maximum(abs, θ)
    nl_rel[k] = sum(abs, nl) / sum(θ .^ 2 ./ 2)
end

p_nl = plot(layout = (2, 1), size = (700, 600), guidefontsize = 14, tickfontsize = 12, legend = false)
plot!(p_nl[1], ϵ_list, θ_max,  marker = :circle, ms = 4, ylabel = L"\max_t |\theta|", xlabel = "")
plot!(p_nl[2], ϵ_list, nl_rel, marker = :circle, ms = 4, ylabel = L"\langle|\cos\theta-1+\theta^2/2|\rangle\,/\,\langle\theta^2/2\rangle", xlabel = L"\epsilon_p", yscale = :log10)
display(p_nl)


In [ ]:
# Same non-linearity measures as above, but averaged over ALL red (non-cat) trajectories at each ϵ_p.
# Each trajectory gets its own max_t |θ| and <|nl|>/<θ²/2>; the plotted curve is the mean over trajectories,
# with the individual trajectories shown as light points to display the spread.

ϵ_list = collect(ϵ_p_list)
t = collect(t_save)

θ_max_all  = [Float64[] for _ in ϵ_list]   # per-trajectory max_t |θ|
nl_rel_all = [Float64[] for _ in ϵ_list]   # per-trajectory <|cos θ - 1 + θ²/2|> / <θ²/2>

for (k, ϵ_p) in enumerate(ϵ_list)
    red_a = Final_all_a[ϵ_p]
    red_b = Final_all_b[ϵ_p]

    for key in keys(red_a)
        a_rot = ComplexF64.(red_a[key])
        b_rot = ComplexF64.(red_b[key])

        θ  = @. 2 * real(-1im * (a_rot * exp(-1im * ω_a * t) + b_rot * exp(-1im * ω_b * t)))
        nl = @. cos(θ) - 1 + θ^2 / 2

        push!(θ_max_all[k],  maximum(abs, θ))
        push!(nl_rel_all[k], sum(abs, nl) / sum(θ .^ 2 ./ 2))
    end
end

# median over red trajectories (NaN where there is none)
median_or_nan(v) = isempty(v) ? NaN : median(v)
θ_max_median  = median_or_nan.(θ_max_all)
nl_rel_median = median_or_nan.(nl_rel_all)

# flattened (ϵ_p, value) pairs for the individual-trajectory points
ϵ_pts     = reduce(vcat, [fill(ϵ_list[k], length(θ_max_all[k])) for k in eachindex(ϵ_list)])
θ_max_pts = reduce(vcat, θ_max_all)
nl_rel_pts = reduce(vcat, nl_rel_all)

p_nl_avg = plot(layout = (2, 1), size = (700, 600), guidefontsize = 14, tickfontsize = 12, legend = false)
scatter!(p_nl_avg[1], ϵ_pts, θ_max_pts, ms = 2, color = :gray, alpha = 0.3)
plot!(p_nl_avg[1], ϵ_list, θ_max_median, marker = :circle, ms = 4, ylabel = L"\max_t |\theta|", xlabel = "")
scatter!(p_nl_avg[2], ϵ_pts, nl_rel_pts, ms = 2, color = :gray, alpha = 0.3)
plot!(p_nl_avg[2], ϵ_list, nl_rel_median, marker = :circle, ms = 4,
      ylabel = L"\langle|\cos\theta-1+\theta^2/2|\rangle\,/\,\langle\theta^2/2\rangle", xlabel = L"\epsilon_p", yscale = :log10)
display(p_nl_avg)


In [ ]:
# Same non-linearity measures as above, but averaged over ALL red (non-cat) trajectories at each ϵ_p.
# Each trajectory gets its own max_t |θ| and <|nl|>/<θ²/2>; the plotted curve is the mean over trajectories,
# with the individual trajectories shown as light points to display the spread.

ϵ_list = collect(ϵ_p_list)
t = collect(t_save)

θ_max_all  = [Float64[] for _ in ϵ_list]   # per-trajectory max_t |θ|
nl_rel_all = [Float64[] for _ in ϵ_list]   # per-trajectory <|cos θ - 1 + θ²/2|> / <θ²/2>

for (k, ϵ_p) in enumerate(ϵ_list)
    red_a = Final_all_a_cat[ϵ_p]
    red_b = Final_all_b_cat[ϵ_p]

    for key in keys(red_a)
        a_rot = ComplexF64.(red_a[key])
        b_rot = ComplexF64.(red_b[key])

        θ  = @. 2 * real(-1im * (a_rot * exp(-1im * ω_a * t) + b_rot * exp(-1im * ω_b * t)))
        nl = @. cos(θ) - 1 + θ^2 / 2

        push!(θ_max_all[k],  maximum(abs, θ))
        push!(nl_rel_all[k], sum(abs, nl) / sum(θ .^ 2 ./ 2))
    end
end

# median over red trajectories (NaN where there is none)
median_or_nan(v) = isempty(v) ? NaN : median(v)
θ_max_median  = median_or_nan.(θ_max_all)
nl_rel_median = median_or_nan.(nl_rel_all)

# flattened (ϵ_p, value) pairs for the individual-trajectory points
ϵ_pts     = reduce(vcat, [fill(ϵ_list[k], length(θ_max_all[k])) for k in eachindex(ϵ_list)])
θ_max_pts = reduce(vcat, θ_max_all)
nl_rel_pts = reduce(vcat, nl_rel_all)

p_nl_avg = plot(layout = (2, 1), size = (700, 600), guidefontsize = 14, tickfontsize = 12, legend = false)
scatter!(p_nl_avg[1], ϵ_pts, θ_max_pts, ms = 2, color = :gray, alpha = 0.3)
plot!(p_nl_avg[1], ϵ_list, θ_max_median, marker = :circle, ms = 4, ylabel = L"\max_t |\theta|", xlabel = "")
scatter!(p_nl_avg[2], ϵ_pts, nl_rel_pts, ms = 2, color = :gray, alpha = 0.3)
plot!(p_nl_avg[2], ϵ_list, nl_rel_median, marker = :circle, ms = 4,
      ylabel = L"\langle|\cos\theta-1+\theta^2/2|\rangle\,/\,\langle\theta^2/2\rangle", xlabel = L"\epsilon_p", yscale = :log10)
display(p_nl_avg)


In [ ]:
# Same non-linearity measures as above, but averaged over ALL red (non-cat) trajectories at each ϵ_p.
# Each trajectory gets its own max_t |θ| and <|nl|>/<θ²/2>; the plotted curve is the mean over trajectories,
# with the individual trajectories shown as light points to display the spread.
ϵ_list = collect(ϵ_p_list)
t = collect(t_save)

θ_max_all  = [Float64[] for _ in ϵ_list]   # per-trajectory max_t |θ|
nl_rel_all = [Float64[] for _ in ϵ_list]   # per-trajectory <|cos θ - 1 + θ²/2|> / <θ²/2>
red_a_all = [[] for _ in ϵ_list]
key_all = [[] for _ in ϵ_list]

for (k, ϵ_p) in enumerate(ϵ_list)
    red_a = Final_all_a_cat[ϵ_p]
    red_b = Final_all_b_cat[ϵ_p]

    for key in keys(red_a)
        a_rot = ComplexF64.(red_a[key])
        b_rot = ComplexF64.(red_b[key])

        θ  = @. 2 * real(-1im * (a_rot * exp(-1im * ω_a * t) + b_rot * exp(-1im * ω_b * t)))
        nl = @. cos(θ) - 1 + θ^2 / 2

        push!(θ_max_all[k],  maximum(abs, θ))
        push!(nl_rel_all[k], sum(abs, nl) / sum(θ .^ 2 ./ 2))
        push!(red_a_all[k], a_rot)
        push!(key_all[k], key)
    end
end
for (k, ϵ_p) in enumerate(ϵ_list)
    red_a = Final_all_a[ϵ_p]
    red_b = Final_all_b[ϵ_p]

    for key in keys(red_a)
        a_rot = ComplexF64.(red_a[key])
        b_rot = ComplexF64.(red_b[key])

        θ  = @. 2 * real(-1im * (a_rot * exp(-1im * ω_a * t) + b_rot * exp(-1im * ω_b * t)))
        nl = @. cos(θ) - 1 + θ^2 / 2

        push!(θ_max_all[k],  maximum(abs, θ))
        push!(nl_rel_all[k], sum(abs, nl) / sum(θ .^ 2 ./ 2))
        push!(red_a_all[k], a_rot)
        push!(key_all[k], key)
    end
end

# median over red trajectories (NaN where there is none)
median_or_nan(v) = isempty(v) ? NaN : median(v)
θ_max_median  = median_or_nan.(θ_max_all)
nl_rel_median = median_or_nan.(nl_rel_all)

# flattened (ϵ_p, value) pairs for the individual-trajectory points
ϵ_pts     = reduce(vcat, [fill(ϵ_list[k], length(θ_max_all[k])) for k in eachindex(ϵ_list)])
θ_max_pts = reduce(vcat, θ_max_all)
nl_rel_pts = reduce(vcat, nl_rel_all)

p_nl_avg = plot(layout = (2, 1), size = (700, 600), guidefontsize = 14, tickfontsize = 12, legend = false)
scatter!(p_nl_avg[1], ϵ_pts, θ_max_pts, ms = 2, color = :gray, alpha = 0.3)
plot!(p_nl_avg[1], ϵ_list, θ_max_median, marker = :circle, ms = 4, ylabel = L"\max_t |\theta|", xlabel = "")
scatter!(p_nl_avg[2], ϵ_pts, nl_rel_pts, ms = 2, color = :gray, alpha = 0.3)
plot!(p_nl_avg[2], ϵ_list, nl_rel_median, marker = :circle, ms = 4,
      ylabel = L"\langle|\cos\theta-1+\theta^2/2|\rangle\,/\,\langle\theta^2/2\rangle", xlabel = L"\epsilon_p", yscale = :log10)
display(p_nl_avg)

In [ ]:
k_index = 14
print(ϵ_list[k_index])
distrib_index = nl_rel_all[k_index]
med = quantile(distrib_index, .9)
idx = argmin(abs.(distrib_index .- med))
print("\n")
print(nl_rel_all[k_index][idx])
traj = red_a_all[k_index][idx]
plot(real(traj), imag(traj))

In [ ]:
t_max = 30/κ_b
re_a_0, im_a_0 = key_all[k_index][idx][1], key_all[k_index][idx][2]
ϕ_a_0, q_a_0=sqrt(2*Z_a)*re_a_0, sqrt(2/Z_a)*im_a_0
re_b_0, im_b_0 = 0, 0
ϕ_b_0, q_b_0=sqrt(2*Z_b)*re_b_0, sqrt(2/Z_b)*im_b_0

system = SystemATS(ω_a, ω_b, E_J, E_Ca, E_Cb, ϕ_a_0, q_a_0, ϕ_b_0, q_b_0)
environment = MeanField(κ_a, κ_b) #OneChain(n, chain_params...)

system_prepared = prepare(system)
environment_prepared = prepare(system,environment)

dt = get_dt(system, environment)
u_dot_0, u_0 = get_init(system_prepared, environment_prepared)
u_0_new = vcat(u_0,u_dot_0)

drive = DriveTwoModesBasics(0.0, ω_d, buffer_phase, 0.0, ω_p, pump_phase)
drive_prepared = prepare(drive)
t_w = 15/κ_b

ϵ_p_i = ϵ_list[k_index]
drive.ϵ_0 = ϵ_p_i
g2_i = besselj(1,ϵ_p_i)*E_J*φ_a^2*φ_b # (Leon's paper)
    
ϵ_d_j = 10*g2_i
drive.ϵ_d = ϵ_d_j
drive_prepared.drive = drive
p = SimulationParams(system_prepared, drive_prepared, environment_prepared)
problem = CoupledODEs(equation_1ODE!, u_0_new, p)
lyapunov(problem, t_max, Δt=dt; d0 = 1e-7, d0_upper = 1e-4, Ttr = t_w)

In [ ]:
# Is the positive Lyapunov exponent of the k_index / idx trajectory real?
# Needs k_index, idx, ϵ_list, key_all from the cells above.

ϵ_k   = ϵ_list[k_index]
key_k = key_all[k_index][idx]
println("ϵ_p = $ϵ_k,  (Re a_0, Im a_0) = $key_k")

# ---------- (A) is the selected trajectory a cat (non-red) one? ----------
println("\n(A) trajectory is in the CAT set: ", haskey(Final_all_a_cat[ϵ_k], key_k))

# ---------- common setup (same physics as the sweep and the Lyapunov cell) ----------
g2_k        = besselj(1, ϵ_k) * E_J * φ_a^2 * φ_b
drive_k     = DriveTwoModesBasics(10 * g2_k, ω_d, buffer_phase, ϵ_k, ω_p, pump_phase)
drive_k_prep = prepare(drive_k)
env_k       = MeanField(κ_a, κ_b)
κ           = κ_b
dt0         = 1 / (50 * ω_a)

# ladder model (used for the Poincaré sweep): u = [a, b]
sys_lad = SystemATS_ladder(ω_a, ω_b, E_J, E_Ca, E_Cb, ComplexF64(key_k[1], key_k[2]), 0.0im)
p_lad   = SimulationParams(prepare(sys_lad), drive_k_prep, prepare(sys_lad, env_k))
u0_lad  = ComplexF64[key_k[1] + 1im * key_k[2], 0.0]
solve_lad(T; alg, tol, saveat, kw...) = solve(ODEProblem(default_equation!, u0_lad, (0.0, T), p_lad), alg;
                                              reltol = tol, abstol = tol, saveat = saveat, kw...)

# second-order model (used for the Lyapunov cell): u = [ϕ_a, ϕ_b, ϕ̇_a, ϕ̇_b]
ϕ_a0, q_a0 = sqrt(2 * Z_a) * key_k[1], sqrt(2 / Z_a) * key_k[2]
sys_2  = SystemATS(ω_a, ω_b, E_J, E_Ca, E_Cb, ϕ_a0, q_a0, 0.0, 0.0)
sys_2p = prepare(sys_2)
env_2p = prepare(sys_2, env_k)
ϕ̇_0, ϕ_0 = get_init(sys_2p, env_2p)
u0_2   = vcat(ϕ_0, ϕ̇_0)
p_2    = SimulationParams(sys_2p, drive_k_prep, env_2p)
solve_2(T; alg, tol, saveat) = solve(ODEProblem(equation_1ODE!, u0_2, (0.0, T), p_2), alg;
                                     reltol = tol, abstol = tol, saveat = saveat)

# ---------- (B) do the two models give the same trajectory? ----------
T_short = 2 / κ
ts_b    = range(0.0, T_short, length = 2000)
sl = solve_lad(T_short; alg = Vern9(), tol = 1e-12, saveat = ts_b)
s2 = solve_2(T_short;   alg = Vern9(), tol = 1e-12, saveat = ts_b)
ϕ_a_lad = [√2 * real(u[1]) for u in sl.u];  ϕ_a_2 = [u[1] for u in s2.u]
ϕ_b_lad = [√2 * real(u[2]) for u in sl.u];  ϕ_b_2 = [u[2] for u in s2.u]
println("\n(B) ladder vs second-order model, t ≤ 2/κ_b (expect ≲ 1e-2: the models differ by a κ² term)")
println("    max|Δϕ_a|/max|ϕ_a| = ", maximum(abs, ϕ_a_lad .- ϕ_a_2) / maximum(abs, ϕ_a_2))
println("    max|Δϕ_b|/max|ϕ_b| = ", maximum(abs, ϕ_b_lad .- ϕ_b_2) / maximum(abs, ϕ_b_2))

# ---------- (C) solver-precision sensitivity of the ladder trajectory, independent of ChaosTools ----------
T_full = 50 / κ
ts_c   = collect(0.0:0.05:T_full)
a_lo   = [u[1] for u in solve_lad(T_full; alg = Tsit5(), tol = 1e-6,  saveat = ts_c, dt = dt0).u]  # as in the sweep
a_hi   = [u[1] for u in solve_lad(T_full; alg = Vern9(), tol = 1e-12, saveat = ts_c).u]
a_hi2  = [u[1] for u in solve_lad(T_full; alg = Vern9(), tol = 1e-13, saveat = ts_c).u]
d_lo   = abs.(a_lo .- a_hi);  d_hi = abs.(a_hi .- a_hi2) .+ 1e-300

div_time(d, thr) = (i = findfirst(>(thr), d); isnothing(i) ? NaN : ts_c[i] * κ)   # in units of 1/κ_b
function growth_rate(d, lo, hi)
    i_end = findfirst(>(hi), d); i_end = isnothing(i_end) ? length(d) : i_end
    m = (d .> lo) .& ((1:length(d)) .< i_end)
    sum(m) < 10 && return NaN
    (hcat(ones(sum(m)), ts_c[m]) \ log.(d[m]))[2]
end
println("\n(C) |Δa| between integrations (time in units of 1/κ_b)")
println("    Tsit5 1e-6  vs Vern9 1e-12: exceeds 0.1 at ", div_time(d_lo, 0.1), "   growth rate λ ≈ ", growth_rate(d_lo, 1e-5, 1e-1), "  (= ", growth_rate(d_lo, 1e-5, 1e-1)/κ, " κ_b)")
println("    Vern9 1e-12 vs Vern9 1e-13: exceeds 0.1 at ", div_time(d_hi, 0.1), "   growth rate λ ≈ ", growth_rate(d_hi, 1e-8, 1e-2), "  (= ", growth_rate(d_hi, 1e-8, 1e-2)/κ, " κ_b)")
println("    (no divergence / NaN ⇒ not chaotic; growth at rate ≈ λ_ChaosTools ⇒ chaotic)")

# ---------- (D) Lyapunov exponent vs solver, tolerance and perturbation size ----------
dt_l = get_dt(sys_2, env_k)
function λ_est(; alg, tol, d0, T = 30 / κ, Ttr = 15 / κ)
    ds = CoupledODEs(equation_1ODE!, u0_2, p_2; diffeq = (alg = alg, abstol = tol, reltol = tol))
    lyapunov(ds, T; Δt = dt_l, d0 = d0, d0_upper = 1e3 * d0, Ttr = Ttr)
end
println("\n(D) Lyapunov exponent (original cell: Tsit5, tol 1e-6, d0 = 1e-7)")
for (name, cfg) in [("Tsit5 1e-6,  d0=1e-7 (original)", (alg = Tsit5(), tol = 1e-6,  d0 = 1e-7)),
                    ("Vern9 1e-10, d0=1e-7",            (alg = Vern9(), tol = 1e-10, d0 = 1e-7)),
                    ("Vern9 1e-12, d0=1e-7",            (alg = Vern9(), tol = 1e-12, d0 = 1e-7)),
                    ("Vern9 1e-12, d0=1e-9",            (alg = Vern9(), tol = 1e-12, d0 = 1e-9)),
                    ("Vern9 1e-12, d0=1e-5",            (alg = Vern9(), tol = 1e-12, d0 = 1e-5)),
                    ("Vern9 1e-12, d0=1e-7, T=15/κ_b",  (alg = Vern9(), tol = 1e-12, d0 = 1e-7, T = 15 / κ)),
                    ("Vern9 1e-12, d0=1e-7, T=60/κ_b",  (alg = Vern9(), tol = 1e-12, d0 = 1e-7, T = 60 / κ))]
    println("    ", rpad(name, 34), " λ = ", λ_est(; cfg...))
end

# ---------- (E) stroboscopic map (system is periodic with period 2π/ω_d since ω_p = 0) ----------
T_d = 2π / (ω_d/2)
t_w = 15 / κ
N   = 400
s_e = solve_2(t_w + N * T_d; alg = Vern9(), tol = 1e-12, saveat = range(t_w, step = T_d, length = N))
pts = reduce(hcat, s_e.u)[[1, 3], :]                       # (ϕ_a, ϕ̇_a)
n_distinct = size(unique(round.(pts ./ maximum(abs, pts, dims = 2); digits = 3), dims = 2), 2)
println("\n(E) stroboscopic points: $N sampled, $n_distinct distinct at 1e-3 relative resolution")
println("    (few points ⇒ periodic orbit; closed curve ⇒ quasi-periodic; ≈ N distinct scattered points ⇒ chaos)")
display(scatter(pts[1, :], pts[2, :], ms = 2, legend = false, xlabel = "ϕ_a", ylabel = "dϕ_a/dt"))
